# Исследование территории с помощью BlocksNet

Ноутбук проводит учебное исследование территории **Магнитогорский городской округ, Челябинская область, Россия**.
Он загружает данные OpenStreetMap, при наличии подключает официальные выгрузки и Microsoft Building Footprints,
нарезает городские кварталы, собирает данные по зданиям и сервисам, строит **автомобильный** граф передвижения,
считает время в пути между кварталами (**в минутах**) и рассчитывает показатели из примера `pipeline.ipynb`.

Оценка населения калибруется по официальной численности Магнитогорского городского округа на 01.01.2025.
Граница, землепользование, здания и сервисы могут быть заменены официальными слоями из папки `better_data`.

Программировать ничего не нужно — просто запускайте ячейки по порядку сверху вниз
(кнопка **Run** над ячейкой или **Run → Run All Cells**).

Данные и результаты сохраняются в файлы и переиспользуются при повторном запуске.


## Порядок работы

1. Настройки, источники и вспомогательные функции (ячейки ниже).
2. Загрузка границы, дорог, воды и нарезка кварталов (раздел 1). После этого — **ОСТАНОВКА**.
3. Землепользование, здания и население, сервисы (разделы 2–4).
4. Автомобильный граф и матрица времени в пути в минутах (раздел 5).
5. Показатели: обеспеченность, плотность, морфотипы, доступность (раздел 6).
6. Сохранение, паспорт источников, проверка результатов и инструкции перезапуска (разделы 7–8).

**Официальные данные.** Положите выгрузки в `data/magnitogorsky_okrug/better_data`:
`boundary.geojson`, `landuse.geojson`, `buildings.geojson`, `services_school.geojson`, `population.csv`.
Если файла нет, используется соответствующий слой OpenStreetMap, а для зданий — дополнение Microsoft.


In [ ]:
# Импорт библиотек и настройки исследования.
import os
import json
import pickle
import time
import warnings

import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt

from data_sources import (
    LAND_USE_RULES,
    OFFICIAL_POPULATION,
    OFFICIAL_POPULATION_DATE,
    OFFICIAL_POPULATION_SOURCE,
    OFFICIAL_SOURCE_REGISTRY,
    calibrate_population,
    combine_buildings,
    download_microsoft_buildings,
    find_official_file,
    infer_microsoft_living,
    load_official_boundary,
    load_official_buildings,
    load_official_landuse,
    load_official_population,
    load_official_services,
    normalize_buildings,
    normalize_functional_zones,
    reclassify_generic_residential,
    write_provenance,
)

warnings.filterwarnings('ignore')

# ================== СЕТЬ / OVERPASS API ==================
# Публичные серверы Overpass иногда перегружены. Поэтому сначала идём на
# публичные зеркала, а NextGIS используем как дополнительный резерв, если
# в переменной окружения или в .env задан NEXTGIS_API_KEY.
#
# Ключ NextGIS берётся из переменной окружения NEXTGIS_API_KEY или из файла
# .env в этой папке (файл .env не попадает в git). Формат строки в .env:
#   NEXTGIS_API_KEY=<ваш ключ>

def load_env_file(path):
    '''Прочитать пары KEY=VALUE из файла .env (если он есть).'''
    if not os.path.exists(path):
        return
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                os.environ.setdefault(key.strip(), value.strip())

for env_path in ['.env', os.path.join('..', '.env')]:
    load_env_file(env_path)

NEXTGIS_API_KEY = os.environ.get('NEXTGIS_API_KEY', '')

OVERPASS_MIRRORS = [
    'https://overpass-api.de/api',
    'https://overpass.kumi.systems/api',
]
if NEXTGIS_API_KEY:
    OVERPASS_MIRRORS.append('https://overpass.nextgis.com/' + NEXTGIS_API_KEY + '/api')
ov_mirror_index = 0

ox.settings.http_user_agent = 'magnitogorsk-research/1.0 (study project)'
ox.settings.requests_timeout = 240
# Отключаем авто-паузу osmnx: на нестандартном сервере она даёт 60 сек на запрос.
ox.settings.overpass_rate_limit = False
ox.settings.overpass_url = OVERPASS_MIRRORS[ov_mirror_index]

# ================== НАСТРОЙКИ ==================
TERRITORY = 'Магнитогорский городской округ, Челябинская область, Россия'
GRAPH_TYPE = 'drive'
FORCE_REDOWNLOAD = False
SERVICE_TYPE = 'school'
USE_MICROSOFT_BUILDINGS = True
OFFICIAL_POPULATION_TARGET = OFFICIAL_POPULATION

# ================== ПАПКИ ==================
DATA_DIR = os.path.join('data', 'magnitogorsky_okrug')
BETTER_DATA_DIR = os.path.join(DATA_DIR, 'better_data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
MS_DIR = os.path.join(RAW_DIR, 'microsoft')
OUTPUT_DIR = 'output'
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')

for folder in [DATA_DIR, BETTER_DATA_DIR, RAW_DIR, MS_DIR, OUTPUT_DIR, FIG_DIR]:
    os.makedirs(folder, exist_ok=True)

landuse_counts = {'osm': 0, 'official': 0}
building_counts = {'osm': 0, 'microsoft': 0, 'official': 0, 'combined': 0}
service_source = 'OpenStreetMap'
calibration = {}
residential_rule = {}
official_population_by_block = None

print('Территория:', TERRITORY)
print('Способ передвижения:', GRAPH_TYPE)
print('Сервер Overpass:', ox.settings.overpass_url)
print('Официальное население:', f'{OFFICIAL_POPULATION_TARGET:,}'.replace(',', ' '), 'чел. на', OFFICIAL_POPULATION_DATE)
print('Рабочие папки созданы:', DATA_DIR, '|', OUTPUT_DIR)


## Источники данных

Официальные выгрузки кладутся в `data/magnitogorsky_okrug/better_data` и имеют приоритет над OSM:

- `boundary.geojson` — официальная граница Магнитогорского городского округа;
- `landuse.geojson` — функциональные зоны, генплан или ПЗЗ;
- `buildings.geojson` — контуры зданий, этажность, жилая площадь;
- `services_school.geojson` — школы и проектная ёмкость;
- `population.csv` — население по кварталам (`block_id`, `population`).

Если официального файла нет, используются OpenStreetMap и Microsoft Building Footprints.
Население всегда приводится к официальной численности округа на 01.01.2025 — 408 487 человек.
Паспорт фактически использованных источников сохраняется в `output/data_provenance.json`.


In [ ]:
# Вспомогательные функции сохранения и загрузки.
# Логика простая: если файл уже есть — читаем его, иначе считаем и сохраняем.

import time

def overpass_call(func):
    '''Вызвать функцию, обращающуюся к Overpass API.

    При сбое (проверка TLS, таймаут, пустой ответ) повторяет запрос и перебирает
    зеркала из OVERPASS_MIRRORS, чтобы обойти перегруженный сервер.
    '''
    global ov_mirror_index
    last_error = None
    for _ in range(len(OVERPASS_MIRRORS) * 3):
        ox.settings.overpass_url = OVERPASS_MIRRORS[ov_mirror_index]
        try:
            return func()
        except Exception as error:
            last_error = error
            print('Overpass', OVERPASS_MIRRORS[ov_mirror_index],
                  '— ошибка:', type(error).__name__, '| пробуем другое зеркало')
            ov_mirror_index = (ov_mirror_index + 1) % len(OVERPASS_MIRRORS)
            time.sleep(5)
    raise last_error

def is_ready(path):
    '''Готов ли файл к переиспользованию.'''
    return os.path.exists(path) and not FORCE_REDOWNLOAD

def save_gdf(gdf, name, folder=OUTPUT_DIR):
    '''Сохранить геослой в GeoJSON.'''
    path = os.path.join(folder, name + '.geojson')
    if os.path.exists(path):
        os.remove(path)
    gdf.to_file(path, driver='GeoJSON')
    print('Сохранено:', path)
    return path

def load_gdf(path):
    '''Прочитать геослой из GeoJSON.'''
    return gpd.read_file(path)

def save_df(df, name, folder=OUTPUT_DIR):
    '''Сохранить таблицу в CSV.'''
    path = os.path.join(folder, name + '.csv')
    df.to_csv(path)
    print('Сохранено:', path)
    return path

def save_fig(fig, name):
    '''Сохранить картинку в output/figures.'''
    path = os.path.join(FIG_DIR, name + '.png')
    fig.savefig(path, bbox_inches='tight', dpi=120)
    print('Сохранено:', path)
    return path

---
## 1. Данные и нарезка кварталов

In [ ]:
# 1.1. Граница территории.
# Сначала ищем официальную границу в better_data, затем готовый файл, затем OSM.

boundaries_path = os.path.join(DATA_DIR, 'boundaries.geojson')
official_boundary = load_official_boundary(BETTER_DATA_DIR)

if official_boundary is not None:
    boundaries = official_boundary
    boundary_source = 'Официальная выгрузка boundary.geojson'
    print('Граница загружена из официальной выгрузки')
elif is_ready(boundaries_path):
    boundaries = load_gdf(boundaries_path)
    boundary_source = 'OpenStreetMap / Nominatim'
    print('Граница загружена из файла')
else:
    boundaries = overpass_call(lambda: ox.geocode_to_gdf(TERRITORY))
    boundaries.to_file(boundaries_path, driver='GeoJSON')
    boundary_source = 'OpenStreetMap / Nominatim'
    print('Граница скачана из OpenStreetMap и сохранена')

# Приводим к метрической системе координат (метры) — так удобнее считать площади.
crs = boundaries.estimate_utm_crs()
boundaries = boundaries.to_crs(crs)
boundaries_4326 = boundaries.to_crs(4326)

print('CRS (система координат):', crs)
print('Источник границы:', boundary_source)
print('Площадь территории, км²:', round(boundaries.area.sum() / 1e6, 1))

ax = boundaries.plot(color='#cce5ff', edgecolor='black')
ax.set_axis_off()
ax.set_title('Граница: ' + TERRITORY)
plt.show()


In [ ]:
# 1.2. Теги OpenStreetMap для дорог, железных дорог и воды.
# Набор тегов такой же, как в примере pipeline.ipynb.

bc_tags = {
    'roads': {
        'highway': ['construction','crossing','living_street','motorway','motorway_link','motorway_junction',
                    'pedestrian','primary','primary_link','raceway','residential','road','secondary','secondary_link',
                    'services','tertiary','tertiary_link','track','trunk','trunk_link','turning_circle',
                    'turning_loop','unclassified'],
        'service': ['living_street','emergency_access'],
    },
    'railways': {'railway': 'rail'},
    'water': {
        'riverbank': True, 'reservoir': True, 'basin': True, 'dock': True, 'canal': True, 'pond': True,
        'natural': ['water','bay'], 'waterway': ['river','canal','ditch'], 'landuse': 'basin', 'water': 'lake',
    },
}

def fetch_osm(name, tags):
    '''Скачать объекты OSM по тегам один раз и сохранить, либо прочитать готовый файл.'''
    path = os.path.join(RAW_DIR, name + '.geojson')
    if is_ready(path):
        gdf = load_gdf(path)
        print(name + ': загружено из файла, объектов', len(gdf))
        return gdf
    gdf = overpass_call(
        lambda: ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags)
    )
    gdf = gdf.reset_index(drop=True).to_crs(crs)
    gdf.to_file(path, driver='GeoJSON')
    print(name + ': скачано из OSM, объектов', len(gdf))
    return gdf

In [ ]:
# Загружаем дороги, железные дороги и воду.
roads = fetch_osm('roads', bc_tags['roads'])
railways = fetch_osm('railways', bc_tags['railways'])
water = fetch_osm('water', bc_tags['water'])

# Оставляем только нужные типы геометрии.
water = water[water.geom_type.isin(['Polygon','MultiPolygon','LineString','MultiLineString'])].copy()
roads = roads[roads.geom_type.isin(['LineString','MultiLineString'])].copy()
railways = railways[railways.geom_type.isin(['LineString','MultiLineString'])].copy()

print('Дороги:', len(roads), '| Ж/д:', len(railways), '| Вода:', len(water))

In [ ]:
# 1.3. Подготовка геометрии и нарезка кварталов.
# Кварталы — это мельчайшие элементы территории (ячейки между улицами, водой и ж/д).

from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

blocks_path = os.path.join(OUTPUT_DIR, 'blocks_raw.geojson')

if is_ready(blocks_path):
    blocks = load_gdf(blocks_path)
    print('Кварталы загружены из файла:', len(blocks))
else:
    lines, polygons = preprocess_urban_objects(roads, railways, water)
    blocks = cut_urban_blocks(boundaries, lines, polygons)
    save_gdf(blocks, 'blocks_raw')
    print('Кварталы нарезаны:', len(blocks))

ax = blocks.plot(color='#dddddd', edgecolor='white', linewidth=0.3)
ax.set_axis_off()
ax.set_title('Городские кварталы (всего ' + str(len(blocks)) + ')')
save_fig(ax.figure, 'blocks')
plt.show()

---
## ⏸ ОСТАНОВИТЕСЬ ЗДЕСЬ

Кварталы подготовлены и сохранены в `output/blocks_raw.geojson`.

**Перед запуском расчётов напишите мне, какие метрики/показатели вы хотите видеть.** Я уточню список
(обеспеченность сервисами, плотность, морфотипы, доступность территории и т.д.) и при необходимости поправлю
следующие ячейки.

Разделы 2–8 можно запускать после этого. Они переиспользуют уже сохранённые файлы и не скачивают данные заново.

---
## 2. Землепользование

In [ ]:
# 2. Функциональные зоны и землепользование.
# Каждому кварталу присваиваем тип использования земли (жильё, бизнес, промышленность и т.д.).

from blocksnet.enums import LandUse
from blocksnet.blocks.assignment import assign_land_use

blocks_lu_path = os.path.join(OUTPUT_DIR, 'blocks.geojson')

if is_ready(blocks_lu_path):
    blocks = load_gdf(blocks_lu_path)
    print('Кварталы с землепользованием загружены из файла:', len(blocks))
else:
    zone_tags = {
        'landuse': True,
        'leisure': ['park', 'garden', 'recreation_ground', 'nature_reserve', 'pitch', 'playground'],
        'amenity': ['school', 'kindergarten', 'hospital', 'clinic', 'university', 'college', 'grave_yard'],
        'natural': ['wood', 'scrub', 'grassland', 'heath'],
    }
    functional_zones = fetch_osm('functional_zones', zone_tags)
    functional_zones = normalize_functional_zones(functional_zones, crs, source='osm')
    landuse_counts['osm'] = len(functional_zones)

    official_zones = load_official_landuse(BETTER_DATA_DIR, crs)
    if official_zones is not None:
        landuse_counts['official'] = len(official_zones)
        functional_zones = pd.concat([official_zones, functional_zones], ignore_index=True)
        print('Добавлены официальные функциональные зоны:', len(official_zones))

    rules = {zone: LandUse[value.split('.')[-1]] for zone, value in LAND_USE_RULES.items()}
    blocks = assign_land_use(blocks, functional_zones, rules)
    save_gdf(blocks, 'blocks')
    print('Землепользование назначено')
    print('Зон OSM:', landuse_counts['osm'], '| официальных зон:', landuse_counts['official'])

ax = blocks.plot(color='#dddddd')
blocks.assign(land_use_label=blocks['land_use'].astype(str)).plot(
    column='land_use_label', legend=True, ax=ax
)
ax.set_axis_off()
ax.set_title('Землепользование кварталов')
save_fig(ax.figure, 'land_use')
plt.show()


---
## 3. Здания и население

In [ ]:
# 3. Здания и население.
# OSM и официальные контуры объединяем, Microsoft добавляем только там, где контура нет.
# Население калибруем по официальной численности городского округа.

from blocksnet.preprocessing.imputing import impute_buildings
from blocksnet.blocks.aggregation import aggregate_objects

bb_path = os.path.join(OUTPUT_DIR, 'buildings_blocks.geojson')

if is_ready(bb_path):
    buildings_blocks = load_gdf(bb_path)
    print('Данные по зданиям загружены из файла:', len(buildings_blocks))
else:
    buildings_osm = fetch_osm('buildings', {'building': True}).to_crs(crs)
    buildings_osm = normalize_buildings(buildings_osm, crs, source='osm')
    building_counts['osm'] = len(buildings_osm)

    official_buildings = load_official_buildings(BETTER_DATA_DIR, crs)
    if official_buildings is not None:
        building_counts['official'] = len(official_buildings)

    microsoft = None
    if USE_MICROSOFT_BUILDINGS:
        microsoft = download_microsoft_buildings(boundaries_4326, MS_DIR, crs)
        residential_area = None
        if 'land_use' in blocks.columns:
            residential_mask = blocks['land_use'].astype(str).str.contains('RESIDENTIAL', na=False)
            if residential_mask.any():
                residential_area = blocks.loc[residential_mask, 'geometry'].union_all()
        microsoft = infer_microsoft_living(microsoft, buildings_osm, residential_area)
        building_counts['microsoft'] = len(microsoft)

    buildings = combine_buildings(buildings_osm, microsoft, official_buildings)
    building_counts['combined'] = len(buildings)
    print('Здания OSM:', building_counts['osm'],
          '| Microsoft:', building_counts['microsoft'],
          '| официальные:', building_counts['official'],
          '| после дедупликации:', building_counts['combined'])

    buildings, generic_reclassified = reclassify_generic_residential(buildings, blocks)
    residential_rule = {
        'method': 'building=yes в жилой зоне, этажность >= 2',
        'min_floors': 2,
        'generic_building_yes_reclassified': generic_reclassified,
    }
    print('Переклассифицировано building=yes в жилые:', generic_reclassified)

    buildings = impute_buildings(buildings, default_living_demand=30)
    buildings_blocks = aggregate_objects(blocks, buildings)[0]
    model_population = float(buildings_blocks['population'].sum())

    official_population_by_block = load_official_population(BETTER_DATA_DIR, blocks)
    if official_population_by_block is not None and official_population_by_block.notna().any():
        buildings_blocks['population_model_raw'] = buildings_blocks['population']
        buildings_blocks['population'] = official_population_by_block.fillna(0)
        calibration = {
            'model_total': model_population,
            'official_total': float(official_population_by_block.sum()),
            'factor': None,
            'blocks_with_population': int((buildings_blocks['population'] > 0).sum()),
            'mode': 'official_by_block',
        }
        print('Население взято из официальной таблицы по кварталам')
    else:
        buildings_blocks, calibration = calibrate_population(
            buildings_blocks, OFFICIAL_POPULATION_TARGET
        )
        calibration['mode'] = 'proportional_calibration'
        print('Население откалибровано по официальной численности округа')

    save_gdf(buildings_blocks, 'buildings_blocks')
    print('Здания обработаны и агрегированы')
    print('Модельная оценка до калибровки:', int(calibration['model_total']))
    print('Коэффициент калибровки:', round(calibration['factor'], 3) if calibration['factor'] else '—')

print('Оценка населения территории:', int(buildings_blocks['population'].sum()))

ax = buildings_blocks.plot('population', legend=True)
ax.set_axis_off()
ax.set_title('Население по кварталам')
save_fig(ax.figure, 'population')
plt.show()


---
## 4. Сервисы

In [ ]:
# 4. Сервисы.
# Для примера берём школы. Официальная выгрузка имеет приоритет над OSM.

from blocksnet.preprocessing.imputing import impute_services

svc_path = os.path.join(OUTPUT_DIR, 'services_blocks_' + SERVICE_TYPE + '.geojson')

if is_ready(svc_path):
    services_blocks = load_gdf(svc_path)
    print('Данные по сервису загружены из файла:', SERVICE_TYPE, len(services_blocks))
else:
    official_services = load_official_services(BETTER_DATA_DIR, SERVICE_TYPE, crs)
    if official_services is not None:
        services = official_services
        service_source = 'Официальная выгрузка'
        print('Сервисы загружены из официальной выгрузки:', len(services))
    else:
        services = fetch_osm(SERVICE_TYPE, {'amenity': SERVICE_TYPE}).to_crs(crs)
        service_source = 'OpenStreetMap'
        print('Сервисы загружены из OpenStreetMap:', len(services))

    services = impute_services(services, SERVICE_TYPE)
    services_blocks = aggregate_objects(blocks, services)[0]
    save_gdf(services_blocks, 'services_blocks_' + SERVICE_TYPE)
    print('Сервис обработан и агрегирован:', SERVICE_TYPE)

print('Источник сервисов:', service_source)
print('Суммарная ёмкость:', int(services_blocks['capacity'].sum()))

ax = services_blocks.plot('capacity', legend=True)
ax.set_axis_off()
ax.set_title('Ёмкость сервиса «' + SERVICE_TYPE + '» по кварталам')
save_fig(ax.figure, 'capacity_' + SERVICE_TYPE)
plt.show()


---
## 5. Время в пути (готовый автомобильный граф)

Для расчёта времени между кварталами используем **автомобильный** граф передвижения.

- **Источник графа:** дорожная сеть OpenStreetMap; её загружает и собирает в граф готовая функция
  `get_accessibility_graph(..., 'drive')` из `blocksnet.relations` (внутри — библиотека IduEdu).
- **Способ передвижения:** автомобиль (`drive`). Интермодальный граф не строим.
- **Единица времени:** минуты (вес ребра `time_min`).

Граф строится один раз и сохраняется в `output/graph_drive.pickle`, затем переиспользуется.

In [ ]:
# 5.1. Автомобильный граф передвижения.

from blocksnet.relations import get_accessibility_graph

graph_path = os.path.join(OUTPUT_DIR, 'graph_' + GRAPH_TYPE + '.pickle')

if is_ready(graph_path):
    with open(graph_path, 'rb') as f:
        graph = pickle.load(f)
    print('Граф загружен из файла:', graph_path)
else:
    territory_geom = blocks.union_all().convex_hull
    territory_gdf = gpd.GeoDataFrame(geometry=[territory_geom], crs=blocks.crs)
    graph = overpass_call(
        lambda: get_accessibility_graph(territory_gdf, GRAPH_TYPE)
    )
    with open(graph_path, 'wb') as f:
        pickle.dump(graph, f)
    print('Граф построен из OpenStreetMap и сохранён:', graph_path)

print('Узлов в графе:', graph.number_of_nodes(), '| рёбер:', graph.number_of_edges())

In [ ]:
# 5.2. Матрица доступности: время в пути между кварталами (в минутах).

from blocksnet.relations import calculate_accessibility_matrix

acc_path = os.path.join(OUTPUT_DIR, 'accessibility_matrix_' + GRAPH_TYPE + '.parquet')

if is_ready(acc_path):
    acc_mx = pd.read_parquet(acc_path)
    print('Матрица доступности загружена из файла')
else:
    acc_mx = calculate_accessibility_matrix(blocks, graph)
    acc_mx.to_parquet(acc_path)
    print('Матрица доступности посчитана и сохранена')

print('Размер матрицы (кварталы × кварталы):', acc_mx.shape)

values = acc_mx.where(acc_mx > 0).stack().dropna()
print('Медианное время в пути, минут:', round(float(values.median()), 1))

ax = values.plot.hist(bins=40, color='#4c72b0')
ax.set_xlabel('Время в пути, минут')
ax.set_title('Распределение времени между кварталами (автомобиль)')
save_fig(ax.figure, 'travel_time_hist')
plt.show()

---
## 6. Показатели (как в pipeline.ipynb)

Считаем те же показатели, что и в примере pipeline: обеспеченность сервисами, плотностные индикаторы,
морфотипы и доступность территории.

### 6.1. Конкурентная обеспеченность сервисом

In [ ]:
# 6.1. Обеспеченность сервисом с учётом конкуренции между кварталами.
# demand и accessibility берём из стандартного конфига BlocksNet для выбранного сервиса.

from blocksnet.config import service_types_config
from blocksnet.analysis.provision import competitive_provision

df = buildings_blocks[['population']].join(services_blocks[['capacity']])

_, demand, accessibility = service_types_config[SERVICE_TYPE].values()
print('Сервис:', SERVICE_TYPE, '| норма спроса:', demand, '| порог доступности, мин:', accessibility)

prov_df, links_df = competitive_provision(df, acc_mx, accessibility, demand)
save_df(prov_df, 'prov_df_' + SERVICE_TYPE)
save_df(links_df, 'links_df_' + SERVICE_TYPE)

ax = blocks.plot(color='#dddddd')
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()
ax.set_title('Обеспеченность сервисом «' + SERVICE_TYPE + '»')
save_fig(ax.figure, 'provision_' + SERVICE_TYPE)
plt.show()

### 6.2. Плотностные индикаторы

In [ ]:
# 6.2. Плотностные индикаторы: fsi, gsi, mxi, l, osr и др.

from blocksnet.analysis.indicators import calculate_density_indicators

buildings_blocks['site_area'] = buildings_blocks.area
density_df = calculate_density_indicators(buildings_blocks)
save_df(density_df, 'density_df')
density_df.head()

### 6.3. Морфотипы застройки

In [ ]:
# 6.3. Морфотипы: Spacematrix и Strelka.

from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)
strelka_df = get_strelka_morphotypes(density_df)

save_df(spacematrix_df, 'spacematrix_df')
save_df(spacematrix_clusters, 'spacematrix_clusters')
save_df(strelka_df, 'strelka_df')

ax = blocks.plot(color='#dddddd')
blocks.join(spacematrix_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()
ax.set_title('Морфотипы Spacematrix')
save_fig(ax.figure, 'spacematrix_morphotypes')
plt.show()

ax = blocks.plot(color='#dddddd')
blocks.join(strelka_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()
ax.set_title('Морфотипы Strelka')
save_fig(ax.figure, 'strelka_morphotypes')
plt.show()

### 6.4. Доступность территории

In [ ]:
# 6.4. Доступность территории (сколько площади достижимо за заданное время).

from blocksnet.analysis.network import area_accessibility

area_acc = area_accessibility(acc_mx, buildings_blocks)
save_df(area_acc, 'area_accessibility')

ax = blocks.join(area_acc).plot('area_accessibility', legend=True)
ax.set_axis_off()
ax.set_title('Доступность территории')
save_fig(ax.figure, 'area_accessibility')
plt.show()

---
## 7. Сохранение и повторное использование

Все слои сохраняются в GeoJSON, таблицы — в CSV, матрица — в parquet, граф — в pickle (папка `output/`).
При повторном запуске файлы читаются заново вместо повторного скачивания и расчёта.

Чтобы пересчитать всё с нуля, поставьте `FORCE_REDOWNLOAD = True` в первой ячейке или удалите папку `output/`.

---
## 8. Проверка результатов

In [ ]:
# Проверяем, что все ключевые файлы созданы, и сохраняем паспорт источников.

expected = [
    os.path.join(OUTPUT_DIR, 'blocks.geojson'),
    os.path.join(OUTPUT_DIR, 'buildings_blocks.geojson'),
    os.path.join(OUTPUT_DIR, 'services_blocks_' + SERVICE_TYPE + '.geojson'),
    os.path.join(OUTPUT_DIR, 'accessibility_matrix_' + GRAPH_TYPE + '.parquet'),
    os.path.join(OUTPUT_DIR, 'prov_df_' + SERVICE_TYPE + '.csv'),
    os.path.join(OUTPUT_DIR, 'density_df.csv'),
    os.path.join(OUTPUT_DIR, 'spacematrix_df.csv'),
    os.path.join(OUTPUT_DIR, 'strelka_df.csv'),
    os.path.join(OUTPUT_DIR, 'area_accessibility.csv'),
    os.path.join(OUTPUT_DIR, 'data_provenance.json'),
]

if landuse_counts['osm'] == 0:
    zones_path = os.path.join(RAW_DIR, 'functional_zones.geojson')
    if os.path.exists(zones_path):
        landuse_counts['osm'] = len(
            normalize_functional_zones(load_gdf(zones_path).to_crs(crs), crs, source='osm')
        )

provenance_path = os.path.join(OUTPUT_DIR, 'data_provenance.json')
previous_provenance = {}
if os.path.exists(provenance_path):
    with open(provenance_path, encoding='utf-8') as file:
        previous_provenance = json.load(file)

if not residential_rule:
    residential_rule = dict(previous_provenance.get('residential_rule') or {})

if not calibration:
    calibration = dict(previous_provenance.get('population') or {})
    if 'population_model_raw' in buildings_blocks.columns:
        model_total = float(buildings_blocks['population_model_raw'].sum())
        calibration.update({
            'model_total': model_total,
            'factor': OFFICIAL_POPULATION_TARGET / model_total if model_total else None,
            'blocks_with_population': int((buildings_blocks['population'] > 0).sum()),
            'mode': 'proportional_calibration',
        })
if not any(building_counts.values()):
    building_counts = dict(previous_provenance.get('buildings') or {})
    if not building_counts:
        building_counts['combined'] = int(buildings_blocks['count'].sum())
if service_source == 'OpenStreetMap' and previous_provenance.get('services', {}).get('source'):
    service_source = previous_provenance['services']['source']

official_inputs = {
    kind: str(find_official_file(BETTER_DATA_DIR, kind))
    for kind in ['boundary', 'landuse', 'buildings', 'population']
    if find_official_file(BETTER_DATA_DIR, kind) is not None
}
if load_official_services(BETTER_DATA_DIR, SERVICE_TYPE, crs) is not None:
    official_inputs['services'] = 'services_' + SERVICE_TYPE

provenance = {
    'territory': TERRITORY,
    'generated_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'boundary': {
        'source': boundary_source,
        'official': official_boundary is not None,
        'area_km2': round(float(boundaries.area.sum()) / 1e6, 1),
        'osm_id': int(boundaries['osm_id'].iloc[0]) if 'osm_id' in boundaries.columns else None,
    },
    'population': {
        'official_total': OFFICIAL_POPULATION_TARGET,
        'official_date': OFFICIAL_POPULATION_DATE,
        'official_source': OFFICIAL_POPULATION_SOURCE,
        **calibration,
        'final_total': int(buildings_blocks['population'].sum()),
    },
    'buildings': building_counts,
    'residential_rule': residential_rule,
    'landuse': {
        **landuse_counts,
        'blocks_with_land_use': int(blocks['land_use'].notna().sum()),
        'blocks_total': int(len(blocks)),
    },
    'services': {'type': SERVICE_TYPE, 'source': service_source},
    'official_inputs': official_inputs,
    'sources': OFFICIAL_SOURCE_REGISTRY,
}
write_provenance(os.path.join(OUTPUT_DIR, 'data_provenance.json'), provenance)

print('Проверка файлов:')
ok = True
for path in expected:
    exists = os.path.exists(path)
    ok = ok and exists
    print(('[ЕСТЬ] ' if exists else '[НЕТ]  ') + path)

print()
print('Все файлы на месте!' if ok else 'Некоторые файлы отсутствуют — проверьте предыдущие ячейки.')
print('Кварталов:', len(blocks))
print('Пустых геометрий:', int(blocks.geometry.is_empty.sum()))
print('Население, чел.:', int(buildings_blocks['population'].sum()))
print('Кварталов с населением:', int((buildings_blocks['population'] > 0).sum()))
print('Кварталов с землепользованием:', int(blocks['land_use'].notna().sum()), 'из', len(blocks))
print('Официальные входы:', official_inputs if official_inputs else 'не найдены')


---
## Как запускать и перезапускать (Windows)

1. Откройте терминал в папке проекта.
2. Активируйте окружение:
   ```
   .venv/Scripts/Activate.ps1
   ```
3. Запустите Jupyter:
   ```
   .venv/Scripts/python.exe -m jupyter lab examples/magnitogorsk_research.ipynb
   ```
4. В меню ноутбука: **Run → Run All Cells** (запустить все ячейки).
5. Первый запуск скачает данные из OpenStreetMap, Microsoft Building Footprints и построит граф — это может занять несколько минут.
   Повторные запуски используют сохранённые файлы и выполняются намного быстрее.

**Официальные выгрузки.** Папка `data/magnitogorsky_okrug/better_data`:
- `boundary.geojson` — граница городского округа;
- `landuse.geojson` — функциональные зоны или ПЗЗ;
- `buildings.geojson` — контуры зданий с этажностью и признаком жилья;
- `services_school.geojson` — школы и их ёмкость;
- `population.csv` — население по кварталам (`block_id`, `population`).

Официальные слои имеют приоритет. Если файла нет, используется OSM; для зданий дополнительно
подключается Microsoft Building Footprints с дедупликацией по вхождениям контуров.

**Пересчитать с нуля:** удалите папку `output/` и файлы в `data/magnitogorsky_okrug/`, либо поставьте
`FORCE_REDOWNLOAD = True` в первой ячейке.

**Сменить город/район:** измените `TERRITORY` в первой ячейке.
**Сменить способ передвижения:** измените `GRAPH_TYPE` на `'walk'` (пешком) — тогда построится пешеходный граф.
